In [14]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, to_timestamp, avg, stddev, lag, round,
    when, row_number, unix_timestamp, lit, to_date, current_timestamp,expr
)
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
from datetime import datetime
import traceback
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.executor.memory", "1g") \
    .config("spark.executor.cores", "1") \
    .config("spark.cores.max", "2") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

In [16]:

spark.sparkContext.setLogLevel("ERROR")

jdbc_url = "jdbc:postgresql://postgres:5432/crypto"

db_properties = {
    "user": "admin",
    "password": "admin",
    "driver": "org.postgresql.Driver"
}

print("\n=== Running Spark Job ===")

run_id = int(time.time())
processing_time_value = datetime.utcnow()

# -----------------------------
# READ ONLY RECENT DATA
# -----------------------------
df = spark.read.parquet("s3a://crypto-data/raw/") \
    .filter(col("event_time") >= current_timestamp() - expr("INTERVAL 20 MINUTES"))

# -----------------------------
# SELECT
# -----------------------------
df = df.select(
    "event_id",
    "event_time",
    "producer_time",
    "producer_id",
    col("id"),
    col("symbol"),
    col("current_price").cast(DoubleType()).alias("price")
)

# -----------------------------
# DEDUP EARLY (small dataset now)
# -----------------------------
df = df.dropDuplicates(["id", "event_time"])

# -----------------------------
# METADATA
# -----------------------------
df = df.withColumn("processing_time", lit(processing_time_value)) \
    .withColumn("run_id", lit(run_id)) \
    .withColumn("lateness_sec",
                unix_timestamp("processing_time") - unix_timestamp("event_time")) \
    .withColumn("ingestion_delay_sec",
                unix_timestamp("processing_time") - unix_timestamp("producer_time")) \
    .withColumn(
        "data_status",
        when(col("lateness_sec") <= 420, "fresh")
        .when(col("lateness_sec") <= 1020, "late")
        .otherwise("too_late")
    )

# -----------------------------
# FILTER VALID DATA
# -----------------------------
df = df.filter(col("data_status") != "too_late")

if df.limit(1).count() == 0:
    print("No usable data. Skipping...")
else:
    coin_window = Window.partitionBy("id").orderBy("event_time")

    # -----------------------------
    # PRICE METRICS
    # -----------------------------
    df = df \
        .withColumn("price_1min_ago", lag("price", 2).over(coin_window)) \
        .withColumn("price_5min_ago", lag("price", 10).over(coin_window)) \
        .withColumn("change_1min",
                    round((col("price") - col("price_1min_ago")) / col("price_1min_ago") * 100, 2)) \
        .withColumn("change_5min",
                    round((col("price") - col("price_5min_ago")) / col("price_5min_ago") * 100, 2))

    # -----------------------------
    # STATS
    # -----------------------------
    df = df \
        .withColumn("SMA", avg("price").over(coin_window.rowsBetween(-4, 0))) \
        .withColumn("volatility", stddev("price").over(coin_window.rowsBetween(-4, 0)))

    # -----------------------------
    # DEDUP (id + event_time)
    # -----------------------------
    df = df.withColumn(
        "row_num",
        row_number().over(
            Window.partitionBy("id", "event_time")
            .orderBy(col("processing_time").desc())
        )
    ).filter(col("row_num") == 1).drop("row_num")

    # -----------------------------
    # LATEST FLAG
    # -----------------------------
    desc_window = Window.partitionBy("id").orderBy(col("event_time").desc())

    df = df.withColumn("rank_desc", row_number().over(desc_window))

    latest_per_coin = df.filter(col("rank_desc") == 1).drop("rank_desc")

    df = df.withColumn("is_latest", col("rank_desc") == 1).drop("rank_desc")

    # -----------------------------
    # TOP GAINERS / LOSERS
    # -----------------------------
    gain_df = latest_per_coin \
        .filter(col("change_5min").isNotNull()) \
        .withColumn("rank", row_number().over(Window.orderBy(col("change_5min").desc()))) \
        .filter(col("rank") <= 5) \
        .select(lit(processing_time_value).alias("processing_time"), "rank", "id", "symbol")

    loss_df = latest_per_coin \
        .filter(col("change_5min").isNotNull()) \
        .withColumn("rank", row_number().over(Window.orderBy(col("change_5min").asc()))) \
        .filter(col("rank") <= 5) \
        .select(lit(processing_time_value).alias("processing_time"), "rank", "id", "symbol")

    df.show()
    df.printSchema()
    latest_per_coin.printSchema()
    gain_df.printSchema()
    loss_df.printSchema()

#         # -----------------------------
#         # WRITE
#         # -----------------------------
#         df.write.jdbc(jdbc_url, "crypto_table", "append", properties=db_properties)

#         latest_per_coin.write.jdbc(jdbc_url, "latest_crypto_temp", "overwrite", properties=db_properties)

#         gain_df.write.jdbc(jdbc_url, "top_5_gainers", "append", properties=db_properties)

#         loss_df.write.jdbc(jdbc_url, "top_5_losers", "append", properties=db_properties)

#         print("Data written to PostgreSQL.")

# except:
#     pass

# except Exception as e:
#     print(f"Error: {e}")
#     traceback.print_exc()

# finally:
#     spark.stop()


=== Running Spark Job ===


[Stage 25:>                                                         (0 + 1) / 1]

+--------------------+--------------------+-------------+--------------------+-----------+------+--------+--------------------+----------+------------+-------------------+-----------+--------------+--------------+-----------+-----------+-------------------+--------------------+---------+
|            event_id|          event_time|producer_time|         producer_id|         id|symbol|   price|     processing_time|    run_id|lateness_sec|ingestion_delay_sec|data_status|price_1min_ago|price_5min_ago|change_1min|change_5min|                SMA|          volatility|is_latest|
+--------------------+--------------------+-------------+--------------------+-----------+------+--------+--------------------+----------+------------+-------------------+-----------+--------------+--------------+-----------+-----------+-------------------+--------------------+---------+
|78e15a75-a139-483...|2026-05-03 15:23:...|         NULL|coingecko-produce...|binancecoin|   bnb|  619.52|2026-05-03 15:32:...|177782

In [7]:
df = spark.read.parquet("s3a://crypto-data/raw/")
df.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- producer_time: timestamp (nullable = true)
 |-- producer_id: string (nullable = true)
 |-- id: string (nullable = true)
 |-- symbol: string (nullable = true)
 |-- current_price: double (nullable = true)
 |-- market_cap: double (nullable = true)
 |-- total_volume: double (nullable = true)
 |-- high_24h: double (nullable = true)
 |-- low_24h: double (nullable = true)
 |-- last_updated: string (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- event_date: date (nullable = true)

